In [ ]:
import pandas as pd
from google.genai import Client
import pandas as pd
from google import genai
from google.genai import types
import google.auth
import os
import json
from pydantic import BaseModel, Field
from typing import Optional

In [ ]:
dummy_data = pd.read_csv("../inputs/sample_llm_answers.csv")

#### Agent Data

In [ ]:

total_calls = len(dummy_data)
avg_token_usage = (dummy_data['input_tokens'] + dummy_data['completion_tokens']).mean()
avg_success_rate = dummy_data['success'].mean()
avg_latency = dummy_data['latency_ms'].mean()

metrics_dict = {
        "total_calls": total_calls,
        "avg_token_usage": round(avg_token_usage, 2),
        "avg_success_rate": round(avg_success_rate,2),
        "avg_latency_ms": round(avg_latency, 2)
    }

#### Judge

In [ ]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "../.venv/credentials/circ_twain.json"
credentials, project = google.auth.default()
print(f"Using service account: {credentials.service_account_email}")
print(f"Project: {project}")

In [ ]:
model_name = "gemini-2.5-flash-lite"
project = 'circulartwain-458407'
location = 'us-central1'


client = genai.Client(
            vertexai=True,
            project=project,
            location=location,
        )


class Judge(BaseModel):
    judgement: str = Field(description="Justification for the scores, noting if tools/reasoning were expected but missing.")
    accuracy: int = Field(ge=1, le=5, description="How factually/technically correct the output is relative to the input.")
    relevance: int = Field(ge=1, le=5, description="How well the response addresses the user's specific intent.")
    reasoning_quality: Optional[int] = Field(None, ge=1, le=5, description="Rate the logic if provided; return null if no reasoning was generated.")
    hallucination: bool = Field(description="True if the model invented information.")

system_instructions = """
You are a Quality Assurance Judge for a Multi-Agent system (OSOM).
Your task is to evaluate an LLM's response based on the provided Input Prompt and Output.

### Evaluation Framework:
Analyze the interaction based on which components are present:

1. **Case A: Direct Completion** (No reasoning, no tools)
   - Focus on **Accuracy** and **Relevance**. 
   - Does the text directly answer the prompt without adding "fluff" or incorrect facts?

2. **Case B: Reasoning/Chain-of-Thought** (Reasoning steps provided)
   - Evaluate the **reasoning_score**. Is the logic sound? 
   - Does the reasoning actually lead to the final answer, or is it disconnected?

3. **Case C: Tool Selection** (MCP tools or function calls involved)
   - Check if the tool choice was logical for the task.
   - If a tool was used, does the **response_text** correctly reflect the tool's intended purpose?

### Specific Constraints:
- **Hallucination**: Mark as TRUE if the model references specific IDs (e.g. Service Order numbers), timestamps, or technical specs that were NOT in the input prompt.
- **Scoring**: Use a 1-5 scale. A '3' is functional but flawed; a '5' is perfect, concise, and logically sound.

### Tone:
Be an objective, technical auditor. Your 'judgement' field must cite specific parts of the input/output to justify your scores.
"""


config = types.GenerateContentConfig(
    temperature=0,
    top_p=0,
    top_k=1,
    max_output_tokens=1000,
    system_instruction=[
        types.Part.from_text(
            text= system_instructions
        )
    ],
    response_mime_type = "application/json",
    response_json_schema = Judge.model_json_schema(),
)

session_id = "a1b2c3d4-e5f6-4a5b-bc6d-7e8f9a0b1c2d"
session_df = dummy_data[dummy_data["session_id"] == session_id].iloc[0]
prompt_parts = []
prompt_parts = [
    f"**User Input**: {session_df['input_prompt']}",
    f"**Model Output**: {session_df['response_text']}"
]

empty_list_keywords = ["none", "null", "[]", "nan"]

reasoning = session_df["reasoning_steps"]

if reasoning and str(reasoning).lower() not in empty_list_keywords:
    prompt_parts.append(f"**Reasoning (Case B)**: {reasoning}")

mcp_tool = session_df["mcp_tool_selected"]
if mcp_tool and str(mcp_tool).lower() not in empty_list_keywords:
    prompt_parts.append(f"**MCP Tool Selected (Case C)**: {mcp_tool}")
    tool_reasoning = session_df["tool_selection_reasoning"]

    if tool_reasoning and str(tool_reasoning).lower() not in empty_list_keywords:
        prompt_parts.append(f"**MCP Selection Logic**: {tool_reasoning}")

prompt = "\n\n".join(prompt_parts)


import time
start_time = time.perf_counter()
response = client.models.generate_content(
            model=model_name,
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=prompt)
                    ]
                )
            ],
            config=config,
        )
end_time = time.perf_counter()
latency_ms = (end_time - start_time) * 1000

judge_metrics = json.loads(response.text)
judge_metrics["latency"] = round(latency_ms, 2)
print(judge_metrics)